# Exploratory Data Analysis (EDA) Report: TempoTune

[cite_start]**Мета аналізу:** Дослідити вплив фізичного контексту (погода, сезон, час доби) на об'єктивні аудіо-характеристики музики у бібліотеках користувачів для побудови контекстно-залежної рекомендаційної системи TempoTune[cite: 2, 43]. 

---

### 1. Розподіл ключових ознак (Feature Distributions)

[cite_start]Аналіз базових аудіо-метрик (Energy, Valence, Acoustic, Dance) підтверджує головну гіпотезу проєкту: фізичний контекст середовища безпосередньо корелює зі звуковим профілем треків, які обирають слухачі[cite: 21, 25].

**Категорія WEATHER:**
* **snowy:** Екстремально високі показники Acoustic на тлі мінімальних Energy та Valence. Сніг чітко асоціюється з повільною, акустичною та меланхолійною музикою.
* **sunny:** Повна протилежність — пікова енергія, найвищий настрій та найнижча акустичність. 
* **rainy та cloudy:** Демонструють схожі розподіли середньої інтенсивності, що створює зону ризику для класифікаторів.

**Категорія SEASON:**
* **summer:** Абсолютний викид у даних. Медіана Energy сягає 80, Dance також на піку, а Acoustic притиснутий до нуля. Це чітко виражений кластер танцювальної та фестивальної музики.
* **winter та autumn:** Демонструють майже ідентичну поведінку з низькою танцювальністю та високою акустичністю, відображаючи перехід користувачів до спокійнішої музики в холодну пору.

**Категорія TIME:**
* **Градієнт настрою:** Метрика Valence лінійно падає від ранку (morning) до ночі (night). 
* **Добові піки:** Найбільш танцювальна музика (Dance) слухається ввечері (evening). В обідній час (afternoon) фіксується аномальний спад Energy та ріст Acoustic, що ймовірно пов'язано з фоновою музикою для концентрації під час роботи чи навчання.

---

### 2. Макро-акустичні профілі (Radar Charts)

Геометрія аудіо-профілів дозволяє оцінити "форму" кожного контексту:
* Найбільшу площу на діаграмах охоплюють summer (серед сезонів) та sunny (серед погоди), розтягуючись до максимальних значень позитивних метрик.
* Профіль snowy виглядає як вузькоспрямований промінь вздовж осі Acoustic.
* Нічний час (night) має найменшу площу серед усіх категорій, що свідчить про загальне радикальне зниження інтенсивності всіх аудіо-ознак.

---

### 3. Аналіз темпу (BPM) та тональностей (Key)

* **BPM як маркер активності:** Категорії summer, sunny, morning та evening утворюють гострі та масивні піки в діапазоні 120-125 BPM (стандартний темп електронної та поп-музики). Інші контексти мають зміщені вліво, більш пласкі розподіли, що вказує на повільніший темп.
* **Тональності як маркер настрою:** Позитивні контексти (sunny, morning) беззаперечно лідирують у світлих мажорних тональностях (C Major, D Major). Водночас night, rainy та snowy показують підвищену концентрацію у мінорних гамах (A Minor, B Minor), що ідеально передає їхній меланхолійний настрій.

---

### 4. Зниження розмірності простору (PCA)

Головна компонента 1 (PCA1) бере на себе близько 45-49% усієї дисперсії в даних, фактично виступаючи мета-індикатором "драйву" пісні.
* **WEATHER:** Проекція демонструє ідеальну просторову роздільність. Кластери sunny (позитивні значення PCA1) та rainy (негативні значення PCA1) формують дві абсолютно незалежні хмари.
* **TIME:** Підкатегорія night впевнено ізолюється в нижній лівий кут графіка, доводячи унікальність свого звучання порівняно з іншими частинами доби.
* **SEASON:** Спостерігається значне накладання класів у центрі, проте winter відчутно зміщується ліворуч, відділяючись від літніх треків.

---

### 💡 Стратегія розробки ML-архітектури TempoTune

[cite_start]Зібрані інсайди дозволяють сформувати чіткий план для етапу моделювання[cite: 33]:

1. **Feature Engineering:**
   * Створити синтетичну ознаку `Energy_to_Acoustic_Ratio`. Вона слугуватиме потужним розділювачем для полярних класів.
   * Трансформувати безперервну метрику BPM у категоріальну ознаку (кошики темпу), що полегшить роботу деревам рішень.
   * Застосувати One-Hot Encoding для ознаки Key, виокремивши бінарну фічу `Is_Major` для покращення розпізнавання емоційного забарвлення.
   * Додати значення PCA1 та PCA2 як нові стовпці у тренувальний набір даних.

2. **Вибір алгоритму:**
   Візуалізація PCA демонструє значне нелінійне накладання класів у категоріях time та season. Лінійні моделі будуть неефективними. [cite_start]Використання ансамблів Boosting (XGBoost, LightGBM) є оптимальним рішенням для знаходження складних нелінійних порогів[cite: 34, 36].

3. **Двоетапна класифікація та Soft Labels:**
   Через сильну подібність деяких станів (cloudy/rainy, autumn/winter), доцільно використовувати архітектуру з двох моделей. Перша модель відсікатиме екстремальні класи (sunny, summer), а друга фокусуватиметься на "нейтральному ядрі". [cite_start]Замість жорсткої класифікації, фінальна модель повинна генерувати ймовірності (`predict_proba()`), що дозволить створювати змішані плейлісти для перехідних погодних чи сезонних станів[cite: 48, 49].